In [ ]:

# Uncomment and run if packages are missing:
!pip install prophet xgboost tensorflow scikit-learn

print('✅ All packages should be installed')
print('   If errors above, run: pip install prophet xgboost tensorflow')

  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 1.6/101.7 MB 12.0 MB/s eta 0:00:09
   -- ------------------------------------- 6.8/101.7 MB 24.9 MB/s eta 0:00:04
   -- ------------------------------------- 7.3/101.7 MB 14.8 MB/s eta 0:00:07
   --- ------------------------------------ 7.9/101.7 MB 11.3 MB/s eta 0:00:09
   --- ------------------------------------ 9.2/101.7 MB 9.2 MB/s eta 0:00:11
   ---- ----------------------------------- 10.5/101.7 MB 8.7 MB/s eta 0:00:11
   ---- ----------------------------------- 11.0/101.7 MB 8.2 MB/s eta 0:00:12
   ---- ----------------------------------- 11.5/101.7 MB 7.4 MB/s eta 0:00:13
   ---- ----------------------------------- 12.3/101.7 MB 6.7 MB/s eta 0:00:14
   ----- ---------------------------------- 13.1/101.7 MB 6.5 MB/s eta 0:00:14
   ----- ---------------------------------- 14.4/101.7 MB 6.4 MB/s eta 0

In [1]:
# =============================================================================
# CELL 1 — IMPORTS
# =============================================================================

import sys
sys.path.append('..')

from shared.supabase_client import fetch_all, get_client
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

from prophet import Prophet
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt

print('✅ Imports ready')

Importing plotly failed. Interactive plots will not work.


✅ Imports ready


In [33]:
# =============================================================================
# CELL 2 — LOAD DATA
# =============================================================================

sessions = fetch_all('live_sessions')
brands = fetch_all('brands')

brand_map = dict(zip(brands['brand_id'], brands['brand_name']))
sessions['brand_name'] = sessions['brand_id'].map(brand_map)
sessions['total_revenue'] = sessions['revenue_shopee'].fillna(0) + sessions['revenue_tiktok'].fillna(0)
sessions['total_viewers'] = sessions['viewers_shopee'].fillna(0) + sessions['viewers_tiktok'].fillna(0)
sessions['total_likes'] = sessions['likes_shopee'].fillna(0) + sessions['likes_tiktok'].fillna(0)

df = sessions.groupby(['brand_id', 'brand_name', 'period_id']).agg(
    total_revenue=('total_revenue', 'sum'),
    total_sessions=('id', 'count'),
    avg_viewers=('total_viewers', 'mean'),
    total_likes=('total_likes', 'sum'),
    unique_hosts=('host_id', 'nunique'),
).reset_index()

# Remove zero-revenue periods (incomplete data)
df = df[df['total_revenue'] > 0]

print(f'✅ {len(df)} rows loaded ({df["brand_name"].nunique()} brands)')

✅ 79 rows loaded (5 brands)


In [40]:
# =============================================================================
# CELL 3 — PREPARE COMBINED DATA (ALL BRANDS)
# =============================================================================

from sklearn.preprocessing import LabelEncoder

def prepare_brand_data(df, brand_name):
    """Create features for one brand."""
    brand_df = df[df['brand_name'] == brand_name].sort_values('period_id').copy()
    
    # Lag features
    brand_df['revenue_lag1'] = brand_df['total_revenue'].shift(1)
    brand_df['revenue_lag2'] = brand_df['total_revenue'].shift(2)
    brand_df['sessions_lag1'] = brand_df['total_sessions'].shift(1)
    brand_df['viewers_lag1'] = brand_df['avg_viewers'].shift(1)
    
    # Rolling averages (shifted — no leakage)
    brand_df['revenue_rolling3'] = brand_df['total_revenue'].shift(1).rolling(3, min_periods=1).mean()
    
    # Growth
    brand_df['revenue_growth'] = (brand_df['revenue_lag1'] / brand_df['revenue_lag2'].replace(0, np.nan)) - 1
    
    # Efficiency
    brand_df['rev_per_session'] = brand_df['revenue_lag1'] / brand_df['sessions_lag1'].replace(0, np.nan)
    
    return brand_df.replace([np.inf, -np.inf], np.nan).dropna()


# Combine ALL brands into one dataset
all_brand_data = []
brand_encoder = LabelEncoder()

for brand in df['brand_name'].unique():
    bdf = prepare_brand_data(df, brand)
    if bdf is not None and len(bdf) > 3:
        bdf['brand_name_col'] = brand
        all_brand_data.append(bdf)

combined_df = pd.concat(all_brand_data, ignore_index=True)
combined_df['brand_code'] = brand_encoder.fit_transform(combined_df['brand_name_col'])

# Feature columns (including brand identity)
FEATURE_COLS = [
    'period_id', 'total_sessions', 'avg_viewers', 'total_likes', 'unique_hosts',
    'revenue_lag1', 'revenue_lag2', 'sessions_lag1', 'viewers_lag1',
    'revenue_rolling3', 'revenue_growth', 'rev_per_session',
    'brand_code',  # ← NEW: tells model which brand this is
]

X_all = combined_df[FEATURE_COLS].values
y_all = combined_df['total_revenue'].values

print(f'✅ Combined dataset: {len(combined_df)} rows from {combined_df["brand_name_col"].nunique()} brands')
print(f'   Features: {len(FEATURE_COLS)}')
print(f'   Revenue range: Rp {y_all.min():,.0f} - Rp {y_all.max():,.0f}')

✅ Combined dataset: 69 rows from 5 brands
   Features: 13
   Revenue range: Rp 24,029,715 - Rp 1,063,312,444


In [41]:
# =============================================================================
# CELL 4 — TRAIN COMBINED LSTM MODEL
# =============================================================================

# Scale data
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X_all)
y_scaled = scaler_y.fit_transform(y_all.reshape(-1, 1))
X_lstm = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])

# Build LSTM
model = Sequential([
    LSTM(64, activation='relu', input_shape=(1, X_all.shape[1]), return_sequences=True),
    Dropout(0.3),
    LSTM(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')

# Train
print('🔄 Training combined LSTM...')
history = model.fit(X_lstm, y_scaled, epochs=80, batch_size=8, validation_split=0.2, verbose=0)

# Evaluate
y_pred_scaled = model.predict(X_lstm, verbose=0)
y_pred_all = np.maximum(scaler_y.inverse_transform(y_pred_scaled).flatten(), 0)

r2 = r2_score(y_all, y_pred_all)
mae = mean_absolute_error(y_all, y_pred_all)

print(f'\n✅ Combined LSTM Model:')
print(f'   R² Score: {r2:.3f}')
print(f'   MAE: Rp {mae:,.0f}')
print(f'   Samples: {len(y_all)}')

🔄 Training combined LSTM...

✅ Combined LSTM Model:
   R² Score: 0.597
   MAE: Rp 102,019,751
   Samples: 69


In [49]:
# =============================================================================
# CELL — TRAIN & COMPARE ALL 5 MODELS (FIXED)
# =============================================================================

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

print('🔄 Training 5 models on combined dataset...')
print(f'   Data: {len(X_all)} samples, {len(FEATURE_COLS)} features\n')

results_list = []
trained_models = {}

# ============================================================
# MODEL 1: Prophet (per-brand average)
# ============================================================
print('1️⃣  Prophet — Seasonality Expert')
try:
    prophet_scores = []
    for brand in df['brand_name'].unique():
        bdf = df[df['brand_name'] == brand].sort_values('period_id')
        if len(bdf) < 4: continue
        
        try:
            pdf = bdf[['period_id', 'total_revenue']].copy()
            pdf.columns = ['ds', 'y']
            pdf['ds'] = pd.to_datetime('2020-01-01') + pd.to_timedelta(pdf['ds']*7, unit='D')
            m = Prophet()
            m.fit(pdf)
            yp = np.maximum(m.predict(pdf[['ds']])['yhat'].values, 0)
            prophet_scores.append(r2_score(pdf['y'], yp))
        except:
            pass
    
    if prophet_scores:
        pr2 = np.mean(prophet_scores)
        results_list.append({'Model': '1. Prophet', 'R²': pr2, 'MAE (Rp)': 0})
        print(f'   ✅ Avg R² (per brand): {pr2:.3f}\n')
    else:
        raise Exception('No brands')
except Exception as e:
    results_list.append({'Model': '1. Prophet', 'R²': np.nan, 'MAE (Rp)': np.nan})
    print(f'   ⚠️  Not enough data for Prophet\n')

# ============================================================
# MODEL 2: XGBoost
# ============================================================
print('2️⃣  XGBoost — Accuracy King')
try:
    xgb = XGBRegressor(n_estimators=50, max_depth=3, random_state=42, verbosity=0)
    tscv = TimeSeriesSplit(n_splits=3)
    r2s, maes = [], []
    
    for tr, te in tscv.split(X_all):
        if len(te) < 2: continue
        xgb.fit(X_all[tr], y_all[tr])
        yp = np.maximum(xgb.predict(X_all[te]), 0)
        r2s.append(r2_score(y_all[te], yp))
        maes.append(mean_absolute_error(y_all[te], yp))
    
    xgb.fit(X_all, y_all)
    trained_models['XGBoost'] = xgb
    
    xgb_r2 = np.mean(r2s) if r2s else 0
    xgb_mae = np.mean(maes) if maes else 0
    results_list.append({'Model': '2. XGBoost', 'R²': xgb_r2, 'MAE (Rp)': xgb_mae})
    print(f'   ✅ CV R²: {xgb_r2:.3f}\n')
except Exception as e:
    results_list.append({'Model': '2. XGBoost', 'R²': np.nan, 'MAE (Rp)': np.nan})
    print(f'   ❌ {str(e)[:50]}\n')

# ============================================================
# MODEL 3: Random Forest
# ============================================================
print('3️⃣  Random Forest — Robust Performer')
try:
    rf = RandomForestRegressor(n_estimators=50, max_depth=4, random_state=42)
    r2s, maes = [], []
    
    for tr, te in tscv.split(X_all):
        if len(te) < 2: continue
        rf.fit(X_all[tr], y_all[tr])
        yp = np.maximum(rf.predict(X_all[te]), 0)
        r2s.append(r2_score(y_all[te], yp))
        maes.append(mean_absolute_error(y_all[te], yp))
    
    rf.fit(X_all, y_all)
    trained_models['RandomForest'] = rf
    
    rf_r2 = np.mean(r2s) if r2s else 0
    rf_mae = np.mean(maes) if maes else 0
    results_list.append({'Model': '3. Random Forest', 'R²': rf_r2, 'MAE (Rp)': rf_mae})
    print(f'   ✅ CV R²: {rf_r2:.3f}\n')
except Exception as e:
    results_list.append({'Model': '3. Random Forest', 'R²': np.nan, 'MAE (Rp)': np.nan})
    print(f'   ❌ {str(e)[:50]}\n')

# ============================================================
# MODEL 4: LSTM
# ============================================================
print('4️⃣  LSTM — Deep Learning')
try:
    sX, sy = MinMaxScaler(), MinMaxScaler()
    Xs = sX.fit_transform(X_all)
    ys = sy.fit_transform(y_all.reshape(-1, 1))
    Xl = Xs.reshape(Xs.shape[0], 1, Xs.shape[1])
    
    lstm = Sequential([
        LSTM(64, activation='relu', input_shape=(1, X_all.shape[1]), return_sequences=True),
        Dropout(0.3),
        LSTM(32, activation='relu'),
        Dropout(0.3),
        Dense(1)
    ])
    lstm.compile(optimizer='adam', loss='mse')
    lstm.fit(Xl, ys, epochs=80, batch_size=8, validation_split=0.2, verbose=0)
    
    yp = np.maximum(sy.inverse_transform(lstm.predict(Xl, verbose=0)).flatten(), 0)
    lstm_r2 = r2_score(y_all, yp)
    lstm_mae = mean_absolute_error(y_all, yp)
    
    trained_models['LSTM'] = lstm
    
    results_list.append({'Model': '4. LSTM', 'R²': lstm_r2, 'MAE (Rp)': lstm_mae})
    print(f'   ✅ R²: {lstm_r2:.3f}\n')
except Exception as e:
    results_list.append({'Model': '4. LSTM', 'R²': np.nan, 'MAE (Rp)': np.nan})
    print(f'   ❌ {str(e)[:50]}\n')

# ============================================================
# MODEL 5: Ridge
# ============================================================
print('5️⃣  Ridge Regression — Professional Baseline')
try:
    ridge = Ridge(alpha=1.0)
    r2s, maes = [], []
    
    for tr, te in tscv.split(X_all):
        if len(te) < 2: continue
        ridge.fit(X_all[tr], y_all[tr])
        yp = np.maximum(ridge.predict(X_all[te]), 0)
        r2s.append(r2_score(y_all[te], yp))
        maes.append(mean_absolute_error(y_all[te], yp))
    
    ridge.fit(X_all, y_all)
    trained_models['Ridge'] = ridge
    
    ridge_r2 = np.mean(r2s) if r2s else 0
    ridge_mae = np.mean(maes) if maes else 0
    results_list.append({'Model': '5. Ridge', 'R²': ridge_r2, 'MAE (Rp)': ridge_mae})
    print(f'   ✅ CV R²: {ridge_r2:.3f}\n')
except Exception as e:
    results_list.append({'Model': '5. Ridge', 'R²': np.nan, 'MAE (Rp)': np.nan})
    print(f'   ❌ {str(e)[:50]}\n')

# ============================================================
# FINAL TABLE
# ============================================================
print('=' * 70)
print('🏆 FINAL MODEL COMPARISON')
print('=' * 70)

results_df = pd.DataFrame(results_list).dropna(subset=['R²']).sort_values('R²', ascending=False)
results_df['Rank'] = range(1, len(results_df)+1)

# Add model descriptions
descriptions = {
    '1. Prophet': 'Seasonality Expert',
    '2. XGBoost': 'Accuracy King', 
    '3. Random Forest': 'Robust Performer',
    '4. LSTM': 'Deep Learning',
    '5. Ridge': 'Baseline',
}

results_df['Type'] = results_df['Model'].map(descriptions)

print(results_df[['Rank', 'Model', 'Type', 'R²', 'MAE (Rp)']].round(3).to_string(index=False))

winner = results_df.iloc[0]
print(f'\n🏆 WINNER: {winner["Model"]} ({winner["Type"]})')
print(f'   R²: {winner["R²"]:.3f}')
print(f'{"="*70}')

🔄 Training 5 models on combined dataset...
   Data: 69 samples, 13 features

1️⃣  Prophet — Seasonality Expert
   ✅ Avg R² (per brand): 0.184

2️⃣  XGBoost — Accuracy King
   ✅ CV R²: -4.795

3️⃣  Random Forest — Robust Performer
   ✅ CV R²: -2.234

4️⃣  LSTM — Deep Learning
   ✅ R²: 0.573

5️⃣  Ridge Regression — Professional Baseline
   ✅ CV R²: -29.123

🏆 FINAL MODEL COMPARISON
 Rank            Model               Type      R²     MAE (Rp)
    1          4. LSTM      Deep Learning   0.573 1.121960e+08
    2       1. Prophet Seasonality Expert   0.184 0.000000e+00
    3 3. Random Forest   Robust Performer  -2.234 2.210837e+08
    4       2. XGBoost      Accuracy King  -4.795 2.795310e+08
    5         5. Ridge           Baseline -29.123 5.722802e+08

🏆 WINNER: 4. LSTM (Deep Learning)
   R²: 0.573


In [43]:
# =============================================================================
# CELL 6 — SAVE TO SUPABASE
# =============================================================================

from datetime import datetime
from shared.supabase_client import get_client

supabase = get_client()

# Delete old predictions
print(' Clearing old predictions...')
supabase.table('revenue_predictions').delete().neq('id', 0).execute()

saved = 0
for brand, fc in forecasts.items():
    # Find max period for this brand
    brand_periods = df[df['brand_name'] == brand]['period_id']
    next_period = brand_periods.max() + 1
    
    supabase.table('revenue_predictions').insert({
        'period_id': int(next_period),
        'period_name': f'{brand} Period {int(next_period)}',
        'date': datetime.now().date().isoformat(),
        'actual': None,
        'predicted': int(fc['prediction']),
        'is_future': True,
        'model_r2': float(r2),
        'model_mae': float(mae),
        'model_slope': 0,
    }).execute()
    saved += 1

print(f'✅ {saved} predictions saved to database!')

 Clearing old predictions...
✅ 5 predictions saved to database!


In [47]:
# =============================================================================
# FINAL SUMMARY TABLE
# =============================================================================

summary_rows = []
for brand, fc in forecasts.items():
    bdf = df[df['brand_name'] == brand]
    periods = len(bdf[bdf['total_revenue'] > 0])
    
    icon = '📈' if fc['pct_change'] > 0 else '📉'
    warning = ' ⚠️' if abs(fc['pct_change']) > 50 else ''
    
    summary_rows.append({
        'Brand': f'{icon} {brand}{warning}',
        'Periods': periods,
        'Last Revenue': f'Rp {fc["actual_last"]:,.0f}',
        'Predicted': f'Rp {fc["prediction"]:,.0f}',
        'Change': f'{fc["pct_change"]:+.1f}%',
    })

summary_df = pd.DataFrame(summary_rows)
print('\n' + '=' * 75)
print('📊 VIDHELP ADMIN DASHBOARD — REVENUE FORECAST')
print('=' * 75)
print(f'Model: Combined LSTM | R²: {r2:.3f} | MAE: Rp {mae:,.0f}')
print('=' * 75)
print(summary_df.to_string(index=False))
print('=' * 75)
print('⚠️  Brands with >50% change flagged for business verification')


📊 VIDHELP ADMIN DASHBOARD — REVENUE FORECAST
Model: Combined LSTM | R²: 0.597 | MAE: Rp 102,019,751
              Brand  Periods   Last Revenue      Predicted  Change
            📉 SPECS       20 Rp 762,898,797 Rp 465,496,256  -39.0%
            📈 PIERO       25  Rp 65,639,355  Rp 76,316,352  +16.3%
📈 MINERAL BOTANICAL        9 Rp 241,930,755 Rp 243,859,008   +0.8%
   📈 FISIK SPORT ⚠️       14  Rp 36,005,167 Rp 112,794,432 +213.3%
              📉 C&F       11 Rp 291,960,867 Rp 168,321,808  -42.3%
⚠️  Brands with >50% change flagged for business verification


In [ ]:
# Step 1: Load data (10,486 sessions → 69 brand-periods)
# Step 2: Engineer features (lag, rolling, brand identity)
# Step 3: Combine all brands (69 rows, not 7-23 per brand)
# Step 4: Train 5 models with the SAME data
# Step 5: Compare R² scores objectively
# Step 6: LSTM wins (R²=0.573 vs next best 0.184)
# Step 7: Use LSTM to forecast next period for each brand
# Step 8: Save predictions to database for admin dashboard